In [ ]:
# =============================================================================
# IMPORTS AND ENVIRONMENT SETUP
# =============================================================================
import os
os.environ["ALBUMENTATIONS_DISABLE_VERSION_CHECK"] = "1"

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import (
    Layer, Conv2D, Dense, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Multiply, Add, Reshape, Concatenate, Activation, Lambda
)
from tensorflow.keras import backend as K
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import albumentations as A
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

sns.set_style('darkgrid')
print("TensorFlow version:", tf.__version__)
print("Albumentations version:", A.__version__)

In [ ]:
# =============================================================================
# CUSTOM ATTENTION BLOCK - IDENTICAL TO TRAINING SCRIPT
# =============================================================================
class AttentionBlock(Layer):
    def __init__(self, attention_type="cbam", reduction_ratio=16, **kwargs):
        super(AttentionBlock, self).__init__(**kwargs)
        self.attention_type = attention_type.lower()
        self.reduction_ratio = reduction_ratio

    def build(self, input_shape):
        filters = input_shape[-1]
        reduced_filters = max(filters // self.reduction_ratio, 1)

        if self.attention_type == "cbam":
            # Channel Attention
            self.channel_dense1 = Dense(reduced_filters, activation='relu', kernel_initializer='he_normal')
            self.channel_dense2 = Dense(filters, kernel_initializer='he_normal')
            self.avg_pool = GlobalAveragePooling2D()
            self.max_pool = GlobalMaxPooling2D()
            # Spatial Attention
            self.spatial_conv = Conv2D(1, (7, 7), padding='same', activation='sigmoid', kernel_initializer='he_normal')

        elif self.attention_type == "self":
            self.query_conv = Conv2D(filters // 8, (1, 1), padding="same")
            self.key_conv = Conv2D(filters // 8, (1, 1), padding="same")
            self.value_conv = Conv2D(filters, (1, 1), padding="same")

    def call(self, inputs):
        if self.attention_type == "cbam":
            # Channel attention
            avg_pool = self.avg_pool(inputs)
            avg_pool = Reshape((1, 1, -1))(avg_pool)
            avg_pool = self.channel_dense1(avg_pool)
            avg_pool = self.channel_dense2(avg_pool)

            max_pool = self.max_pool(inputs)
            max_pool = Reshape((1, 1, -1))(max_pool)
            max_pool = self.channel_dense1(max_pool)
            max_pool = self.channel_dense2(max_pool)

            channel_attention = Add()([avg_pool, max_pool])
            channel_attention = Activation('sigmoid')(channel_attention)
            x = Multiply()([inputs, channel_attention])

            # Spatial attention
            avg_pool_spatial = Lambda(lambda z: K.mean(z, axis=-1, keepdims=True))(x)
            max_pool_spatial = Lambda(lambda z: K.max(z, axis=-1, keepdims=True))(x)
            concat_spatial = Concatenate(axis=-1)([avg_pool_spatial, max_pool_spatial])
            spatial_attention = self.spatial_conv(concat_spatial)
            x = Multiply()([x, spatial_attention])
            return x

        elif self.attention_type == "self":
            batch_size, h, w, c = tf.shape(inputs)[0], inputs.shape[1], inputs.shape[2], inputs.shape[3]
            query = self.query_conv(inputs)
            key = self.key_conv(inputs)
            value = self.value_conv(inputs)
            query = tf.reshape(query, (batch_size, h*w, -1))
            key = tf.reshape(key, (batch_size, h*w, -1))
            value = tf.reshape(value, (batch_size, h*w, -1))
            attention = tf.nn.softmax(tf.matmul(query, key, transpose_b=True) / tf.sqrt(tf.cast(c, tf.float32)))
            out = tf.matmul(attention, value)
            out = tf.reshape(out, (batch_size, h, w, -1))
            return inputs + out

In [ ]:
# =============================================================================
# LOAD PRE-TRAINED MODEL
# =============================================================================
MODEL_PATH = 'ag_densenet_cbam.h5'  # Update with your actual model path

model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={'AttentionBlock': AttentionBlock},
    compile=False
)
print("✅ Model loaded successfully.")
model.summary()

In [ ]:
# =============================================================================
# CUSTOM METAL STREAK SIMULATION
# =============================================================================
def add_metal_streak(image, intensity=0.7, num_streaks=2):
    """
    Simulate metal streak artifacts (radiopaque lines).
    image: uint8 numpy array (H, W, C)
    """
    img = image.copy()
    h, w = img.shape[:2]
    for _ in range(np.random.randint(1, num_streaks + 1)):
        x1, y1 = np.random.randint(0, w), np.random.randint(0, h)
        x2, y2 = np.random.randint(0, w), np.random.randint(0, h)
        thickness = np.random.randint(2, 6)
        brightness = np.random.randint(180, 255)
        color = (brightness, brightness, brightness)
        cv2.line(img, (x1, y1), (x2, y2), color, thickness)
        img = cv2.GaussianBlur(img, (3, 3), 0)
    return img

class MetalStreakTransform:
    def __init__(self, p=1.0):
        self.p = p
    def __call__(self, **kwargs):
        if np.random.random() < self.p:
            return {'image': add_metal_streak(kwargs['image'])}
        return kwargs

print("✅ Metal streak transform defined.")

In [ ]:
# =============================================================================
# ARTIFACT TRANSFORM PIPELINES
# =============================================================================
def get_transform(artifact_type, p=1.0):
    if artifact_type == 'clean':
        return A.Compose([])
    elif artifact_type == 'motion_blur':
        return A.Compose([A.MotionBlur(blur_limit=(7, 15), p=p)])
    elif artifact_type == 'gaussian_noise':
        return A.Compose([A.GaussNoise(var_limit=(50.0, 200.0), p=p)])
    elif artifact_type == 'metal_streak':
        return A.Compose([MetalStreakTransform(p=p)])
    else:
        raise ValueError(f"Unknown artifact type: {artifact_type}")

print("✅ Transform pipelines ready.")

In [ ]:
# =============================================================================
# VISUALIZE SAMPLE IMAGES WITH DIFFERENT ARTIFACTS
# =============================================================================
def visualize_artifacts(test_dir, num_samples=3):
    datagen = ImageDataGenerator(rescale=1.0/255.0)
    gen = datagen.flow_from_directory(
        test_dir, target_size=(224,224), batch_size=1,
        class_mode='categorical', shuffle=True, color_mode='rgb'
    )
    
    artifact_types = ['clean', 'motion_blur', 'gaussian_noise', 'metal_streak']
    fig, axes = plt.subplots(num_samples, len(artifact_types), figsize=(15, 4*num_samples))
    
    for i in range(num_samples):
        img_batch, _ = next(gen)
        img_uint8 = (img_batch[0] * 255).astype(np.uint8)
        
        for j, art in enumerate(artifact_types):
            transform = get_transform(art)
            aug = transform(image=img_uint8)['image']
            axes[i, j].imshow(aug)
            axes[i, j].set_title(f"{art}" if i == 0 else "")
            axes[i, j].axis('off')
    
    plt.tight_layout()
    plt.savefig('artifact_samples.png', dpi=300, bbox_inches='tight')
    plt.show()

# Run visualization
TEST_DIR = 'data/test'  # Update path if needed
visualize_artifacts(TEST_DIR, num_samples=2)

In [ ]:
# =============================================================================
# EVALUATION FUNCTION WITH ON-THE-FLY ARTIFACT APPLICATION
# =============================================================================
def evaluate_on_artifact(model, test_dir, artifact_type, img_size=(224,224), batch_size=8):
    """
    Evaluate model on test set with specified artifact applied.
    Returns dictionary of metrics and confusion matrix.
    """
    # Create generator with custom preprocessing that applies artifact
    def preprocess(img):
        img_np = np.array(img)
        transform = get_transform(artifact_type)
        aug = transform(image=img_np)['image']
        return aug.astype(np.float32) / 255.0
    
    datagen = ImageDataGenerator(preprocessing_function=preprocess)
    generator = datagen.flow_from_directory(
        test_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False,
        color_mode='rgb'
    )
    
    y_true = generator.classes
    y_pred_probs = model.predict(generator, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted')
    rec = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    cm = confusion_matrix(y_true, y_pred)
    
    if cm.shape == (2, 2):
        TN, FP, FN, TP = cm.ravel()
        spec = TN / (TN + FP)
    else:
        spec = None
    
    return {
        'artifact': artifact_type,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'specificity': spec,
        'f1': f1,
        'confusion_matrix': cm,
        'y_true': y_true,
        'y_pred': y_pred
    }

print("✅ Evaluation function defined.")

In [ ]:
# =============================================================================
# EXECUTE EVALUATION FOR EACH ARTIFACT TYPE
# =============================================================================
artifact_types = ['clean', 'motion_blur', 'gaussian_noise', 'metal_streak']
results = {}

for art in tqdm(artifact_types, desc="Evaluating artifacts"):
    print(f"\n🔍 Testing on: {art.upper()}")
    res = evaluate_on_artifact(model, TEST_DIR, art, batch_size=8)
    results[art] = res
    print(f"   Accuracy: {res['accuracy']:.4f}")
    print(f"   Precision: {res['precision']:.4f}")
    print(f"   Recall: {res['recall']:.4f}")
    if res['specificity']:
        print(f"   Specificity: {res['specificity']:.4f}")
    print(f"   F1 Score: {res['f1']:.4f}")

print("\n✅ All evaluations complete.")

In [ ]:
# =============================================================================
# CREATE SUMMARY TABLE
# =============================================================================
df_results = pd.DataFrame([
    {
        'Artifact': art,
        'Accuracy': res['accuracy'],
        'Precision': res['precision'],
        'Recall': res['recall'],
        'Specificity': res['specificity'] if res['specificity'] else np.nan,
        'F1-Score': res['f1']
    }
    for art, res in results.items()
])

clean_acc = df_results.loc[df_results['Artifact'] == 'clean', 'Accuracy'].values[0]
df_results['Drop (%)'] = ((clean_acc - df_results['Accuracy']) * 100).round(2)

print("=" * 60)
print("ROBUSTNESS EVALUATION SUMMARY")
print("=" * 60)
print(df_results.to_string(index=False))

# Save to CSV
df_results.to_csv('robustness_results.csv', index=False)
print("\n✅ Results saved to 'robustness_results.csv'")

In [ ]:
# =============================================================================
# PLOT CONFUSION MATRICES FOR EACH ARTIFACT TYPE
# =============================================================================
class_names = ['Normal', 'Subluxation']
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for idx, (art, res) in enumerate(results.items()):
    cm = res['confusion_matrix']
    ax = axes[idx]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, cbar=False)
    ax.set_title(f"{art.replace('_', ' ').title()}")
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices_all_artifacts.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# BAR PLOT: ACCURACY DROP ACROSS ARTIFACT TYPES
# =============================================================================
plt.figure(figsize=(10, 6))
colors = ['#2ecc71' if art == 'clean' else '#e74c3c' for art in df_results['Artifact']]
bars = plt.bar(df_results['Artifact'], df_results['Accuracy'], color=colors, alpha=0.8)
plt.axhline(y=clean_acc, color='green', linestyle='--', label=f'Clean Baseline ({clean_acc:.3f})')
plt.ylabel('Accuracy')
plt.title('Model Robustness Across Clinical Artifacts')
plt.ylim(0.7, 1.0)
plt.legend()

# Annotate bars with values
for bar, acc in zip(bars, df_results['Accuracy']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('accuracy_drop_artifacts.png', dpi=300, bbox_inches='tight')
plt.show()